# Negative-binomial re-analysis: egress exits vs. wildfire fatalities

Reruns the count model from [egress-fatalities-reanalysis.md](egress-fatalities-reanalysis.md) on the
corrected dataset (`data/fire-fatality-corrected.csv`, 38 fire × census-place records).

**Model:** `deaths ~ exits + offset(log population)`, negative binomial (NB2, α estimated by ML).
The offset turns the count model into a *rate* model, and NB handles the heavy overdispersion that
makes Poisson (and OLS on per-capita ratios) inappropriate here.

Sections: data → NB fit (original vs corrected denominators) → fitted rate curve → the paper's
Fig S4 OLS for comparison → leave-one-out jackknife.

In [1]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.simplefilter("ignore", RuntimeWarning)  # statsmodels NB llf eval emits benign log(0) warnings

df = pd.read_csv("data/fire-fatality-corrected.csv")
df["label"] = df.Fire.str.title() + " / " + df.census_place
print(f"{len(df)} records, {df.census_place.nunique()} unique places, {df.Identified_fatalities.sum()} deaths")
df.sort_values("Identified_fatalities", ascending=False).head(10)

38 records, 35 unique places, 312 deaths


,Fire,census_place,Identified_fatalities,exits,population_orig,population_corrected,fatality_pct_corrected,label
0,Hawaii/Lahaina,Lahiaina,102,4.0,13261,13261,0.7692,Hawaii/Lahaina / Lahiaina
1,CAMP,Paradise,66,6.0,7730,26800,0.2463,Camp / Paradise
2,Eaton,Altadena,19,20.0,41921,41921,0.0453,Eaton / Altadena
3,NORTH COMPLEX,Berry Creek,13,3.0,1190,1190,1.0924,North Complex / Berry Creek
4,2016 Great Smoky Mountain wildfires,Gatlinburg,12,9.0,3684,3684,0.3257,2016 Great Smoky Mountain Wildfires / Gatlinburg
5,TUBBS,Larkfield-Wikiup CDP,11,11.0,7595,7595,0.1448,Tubbs / Larkfield-Wikiup CDP
6,CAMP,Concow,9,6.0,306,710,1.2676,Camp / Concow
7,CAMP,Magalia,9,10.0,10537,10537,0.0854,Camp / Magalia
8,REDWOOD VALLEY,Redwood Valley,9,12.0,1798,1798,0.5006,Redwood Valley / Redwood Valley
9,Palisades,Malibu,5,14.5,10915,10915,0.0458,Palisades / Malibu


## 1. Negative binomial fit — original vs. corrected denominators

Six populations in the source data were post-fire/depopulated census figures or geographically
misassigned, all at low-exit places, all inflating the fatality rate toward the paper's conclusion.
We fit the NB model with both the original and corrected denominators.

In [2]:
def fit_nb(pop_col):
    return smf.negativebinomial(
        "Identified_fatalities ~ exits", data=df, offset=np.log(df[pop_col])
    ).fit(disp=0)

fits = {label: fit_nb(col) for label, col in
        [("original", "population_orig"), ("corrected", "population_corrected")]}

rows = []
for label, m in fits.items():
    b, (lo, hi) = m.params["exits"], m.conf_int().loc["exits"]
    rows.append({
        "denominators": label,
        "coef (exits)": round(b, 4),
        "IRR / exit": round(np.exp(b), 3),
        "effect / exit": f"{(np.exp(b) - 1) * 100:+.1f}%",
        "95% CI (IRR)": f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]",
        "p": f"{m.pvalues['exits']:.1e}",
        "alpha": round(m.params["alpha"], 3),
    })
pd.DataFrame(rows).set_index("denominators")

,coef (exits),IRR / exit,effect / exit,95% CI (IRR),p,alpha
denominators,,,,,,
original,-0.2075,0.813,-18.7%,"[0.754, 0.875]",4.8e-08,1.127
corrected,-0.1785,0.837,-16.3%,"[0.777, 0.900]",1.9e-06,1.047


In [3]:
# Why NB and not Poisson: Pearson dispersion under Poisson is ~12x what equidispersion allows
pois = smf.glm("Identified_fatalities ~ exits", data=df, family=sm.families.Poisson(),
               offset=np.log(df.population_corrected)).fit()
print(f"Poisson Pearson dispersion phi = {pois.pearson_chi2 / pois.df_resid:.1f}  (equidispersion -> 1)")

Poisson Pearson dispersion phi = 12.3  (equidispersion -> 1)


## 2. Fitted rate curve

Observed per-capita fatality rate (corrected denominators, log scale) with the NB-implied rate
`exp(b0 + b1·exits)`. The decline is smooth and constant-per-exit — **no threshold/kink at ~6 exits**.
Marker area ∝ population; hover for fire/place.

In [4]:
m = fits["corrected"]
df["rate_pct"] = df.Identified_fatalities / df.population_corrected * 100

fig = px.scatter(
    df, x="exits", y="rate_pct", size="population_corrected", hover_name="label",
    hover_data={"Identified_fatalities": True, "population_corrected": ":,",
                "rate_pct": ":.3f", "exits": True},
    size_max=40, log_y=True, opacity=0.75,
    labels={"exits": "egress exits", "rate_pct": "fatalities per 100 residents (log)"},
)
xs = np.linspace(df.exits.min(), df.exits.max(), 100)
pred = np.exp(m.params["Intercept"] + m.params["exits"] * xs) * 100
fig.add_trace(go.Scatter(x=xs, y=pred, mode="lines", name="NB fitted rate",
                         line=dict(color="firebrick", width=2)))
fig.update_layout(title="Wildfire fatality rate vs egress exits (corrected denominators)",
                  height=480, legend=dict(orientation="h", y=1.08))
fig.show()

## 3. The paper's Fig S4 spec (OLS on per-capita rate), for comparison

OLS on a noisy ratio is the wrong model here, but it's what the paper reported (p = 0.048, our
unweighted rerun p = 0.036). On corrected denominators it slips below conventional significance.

In [5]:
rows = []
for label, col in [("original", "population_orig"), ("corrected", "population_corrected")]:
    rate = df.Identified_fatalities / df[col] * 100
    ols = smf.ols("rate ~ exits", data=df.assign(rate=rate)).fit()
    rows.append({"denominators": label, "slope": round(ols.params["exits"], 4),
                 "p": round(ols.pvalues["exits"], 4), "R^2": round(ols.rsquared, 3)})
pd.DataFrame(rows).set_index("denominators")

,slope,p,R^2
denominators,,,
original,-0.0534,0.0362,0.116
corrected,-0.0386,0.0601,0.095


## 4. Leave-one-out jackknife

Refit the corrected-denominator NB model 38 times, dropping one record each time. If any single
community were driving the result, its deletion would move the coefficient toward zero.

In [6]:
jk = []
for i in df.index:
    sub = df.drop(i)
    mi = smf.negativebinomial("Identified_fatalities ~ exits", data=sub,
                              offset=np.log(sub.population_corrected)).fit(disp=0)
    jk.append({"dropped": df.loc[i, "label"], "coef": mi.params["exits"]})
jk = pd.DataFrame(jk).sort_values("coef").reset_index(drop=True)
full_coef = fits["corrected"].params["exits"]
print(f"full-model coef = {full_coef:.4f}; LOO range [{jk.coef.min():.4f}, {jk.coef.max():.4f}]; "
      f"all negative: {(jk.coef < 0).all()}")

fig = px.scatter(jk, x="coef", y="dropped", height=720,
                 labels={"coef": "exits coefficient after dropping this record", "dropped": ""})
fig.add_vline(x=full_coef, line_dash="dash", line_color="firebrick",
              annotation_text="full model", annotation_position="top")
fig.add_vline(x=0, line_color="gray")
fig.update_layout(title="Jackknife: NB exits coefficient, leave-one-out",
                  yaxis=dict(tickfont=dict(size=9)))
fig.show()

full-model coef = -0.1785; LOO range [-0.1960, -0.1679]; all negative: True


## 5. Robustness

Six checks, in roughly increasing order of severity. The headline asymptotic p-value (~2×10⁻⁶) deserves
skepticism with n = 38, a heavy-tailed exits distribution, and selection on fatal events — the honest
question is whether the *effect* survives, and at what strength.

**5a. Specification battery.** Same mean model, different likelihoods. The zero-truncated NB matters
most: the dataset only contains places with ≥1 death, so the untruncated likelihood is mildly
misspecified by construction.

In [7]:
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

df["lpop"] = np.log(df.population_corrected)
y, X = df.Identified_fatalities, sm.add_constant(df[["exits"]])

specs = {}
specs["NB2 ML (baseline)"] = (m.params["exits"], m.pvalues["exits"])
pois_hc3 = smf.glm("Identified_fatalities ~ exits", df, family=sm.families.Poisson(),
                   offset=df.lpop).fit(cov_type="HC3")
specs["Poisson, HC3 robust SE"] = (pois_hc3.params["exits"], pois_hc3.pvalues["exits"])
glmnb = smf.glm("Identified_fatalities ~ exits", df,
                family=sm.families.NegativeBinomial(alpha=m.params["alpha"]), offset=df.lpop).fit()
specs["GLM-NB (alpha fixed at ML)"] = (glmnb.params["exits"], glmnb.pvalues["exits"])
ztnb = TruncatedLFNegativeBinomialP(y, X, offset=df.lpop, truncation=0).fit(disp=0, maxiter=500)
specs["Zero-truncated NB (selection-honest)"] = (ztnb.params["exits"], ztnb.pvalues["exits"])

pd.DataFrame(
    [{"specification": k, "IRR / exit": round(np.exp(b), 3),
      "effect / exit": f"{(np.exp(b) - 1) * 100:+.1f}%", "p": f"{p:.1e}"}
     for k, (b, p) in specs.items()]
).set_index("specification")

,IRR / exit,effect / exit,p
specification,,,
NB2 ML (baseline),0.837,-16.3%,1.9e-06
"Poisson, HC3 robust SE",0.863,-13.7%,1.4e-04
GLM-NB (alpha fixed at ML),0.837,-16.3%,1.1e-07
Zero-truncated NB (selection-honest),0.852,-14.8%,2.7e-02


**5b. Functional form — is there any threshold?** If the paper's "~6 exits" breakpoint were real,
a hinge term at 6 should beat the smooth linear model. It doesn't: AIC is a statistical tie and the
hinge term is not significant.

In [8]:
df["hinge6"] = np.maximum(df.exits - 6, 0)
forms = {
    "linear (baseline)": m,
    "log(exits)": smf.negativebinomial("Identified_fatalities ~ np.log(exits)", df, offset=df.lpop).fit(disp=0),
    "quadratic": smf.negativebinomial("Identified_fatalities ~ exits + I(exits**2)", df, offset=df.lpop).fit(disp=0),
    "hinge at 6 exits": smf.negativebinomial("Identified_fatalities ~ exits + hinge6", df, offset=df.lpop).fit(disp=0),
}
print(pd.DataFrame([{"form": k, "AIC": round(v.aic, 2), "loglik": round(v.llf, 2)} for k, v in forms.items()])
      .set_index("form").to_string())
print(f"\nhinge term:     coef = {forms['hinge at 6 exits'].params['hinge6']:+.3f}, "
      f"p = {forms['hinge at 6 exits'].pvalues['hinge6']:.3f}")
print(f"quadratic term: p = {forms['quadratic'].pvalues['I(exits ** 2)']:.3f}")

                      AIC  loglik
form                             
linear (baseline)  228.42 -111.21
log(exits)         233.99 -113.99
quadratic          228.98 -110.49
hinge at 6 exits   228.36 -110.18

hinge term:     coef = -0.310, p = 0.155
quadratic term: p = 0.230


**5c. Offset relaxation.** The offset forces deaths ∝ population (elasticity 1). Freeing it tests
whether that assumption manufactures the effect. The fitted elasticity is ~0.6 (significantly < 1) and
the exits effect attenuates to −9.4%/exit but stays significant — some of the baseline effect rides on
the proportionality assumption, the direction doesn't.

In [9]:
free = smf.negativebinomial("Identified_fatalities ~ exits + lpop", df).fit(disp=0)
t = free.t_test("lpop = 1")
b, p = free.params["exits"], free.pvalues["exits"]
print(f"exits:    IRR = {np.exp(b):.3f} ({(np.exp(b)-1)*100:+.1f}%/exit), p = {p:.4f}")
print(f"log(pop): elasticity = {free.params['lpop']:.3f}, H0 elasticity=1: p = {t.pvalue.item():.4f}")

exits:    IRR = 0.906 (-9.4%/exit), p = 0.0112
log(pop): elasticity = 0.601, H0 elasticity=1: p = 0.0004


**5d. Block deletions.** The jackknife (§4) drops one record at a time; here we delete whole
suspect blocks: the two mass-casualty fires (Lahaina + Paradise are 168 of 312 deaths), all six
corrected-denominator rows, and the tiny CDPs whose populations are barely benchmarkable.

In [10]:
def refit(sub):
    mm = smf.negativebinomial("Identified_fatalities ~ exits", sub,
                              offset=np.log(sub.population_corrected)).fit(disp=0)
    return mm.params["exits"], mm.pvalues["exits"]

blocks = {
    "full data (baseline)": df,
    "drop Lahaina + Paradise (168/312 deaths)": df[~df.census_place.isin(["Lahiaina", "Paradise"])],
    "drop the 6 corrected-denominator rows": df[df.population_orig == df.population_corrected],
    "drop tiny CDPs (pop < 500)": df[df.population_corrected >= 500],
}
pd.DataFrame(
    [{"subset": k, "n": len(v), "IRR / exit": round(np.exp(refit(v)[0]), 3),
      "effect / exit": f"{(np.exp(refit(v)[0]) - 1) * 100:+.1f}%", "p": f"{refit(v)[1]:.1e}"}
     for k, v in blocks.items()]
).set_index("subset")

,n,IRR / exit,effect / exit,p
subset,,,,
full data (baseline),38,0.837,-16.3%,1.9e-06
drop Lahaina + Paradise (168/312 deaths),36,0.838,-16.2%,6.4e-06
drop the 6 corrected-denominator rows,32,0.829,-17.1%,6.3e-07
drop tiny CDPs (pop < 500),30,0.867,-13.3%,2.3e-04


**5e. Cluster bootstrap by fire + permutation test.** Records cluster within fires (the Camp fire
alone contributes three places), so we resample *fires* with replacement. And because asymptotic
p-values are optimistic at n = 38, we also build the exact null by permuting exits. **This is the most
important check in the notebook:** the permutation p is ~0.04 — three orders of magnitude weaker than
the asymptotic 2×10⁻⁶, though still under 0.05.

In [11]:
from plotly.subplots import make_subplots

rng = np.random.default_rng(42)
fires = df.Fire.unique()
boot = []
for _ in range(1000):
    bs = pd.concat([df[df.Fire == f] for f in rng.choice(fires, size=len(fires), replace=True)])
    try:
        bm = smf.negativebinomial("Identified_fatalities ~ exits", bs,
                                  offset=np.log(bs.population_corrected)).fit(disp=0, maxiter=200)
        if bm.mle_retvals.get("converged"):
            boot.append(bm.params["exits"])
    except Exception:
        pass
boot = np.array(boot)

perm = []
for _ in range(2000):
    d2 = df.assign(pex=rng.permutation(df.exits.values))
    try:
        pm = smf.negativebinomial("Identified_fatalities ~ pex", d2, offset=df.lpop).fit(disp=0, maxiter=200)
        perm.append(pm.params["pex"])
    except Exception:
        pass
perm = np.array(perm)

b_obs = m.params["exits"]
ci = np.exp(np.percentile(boot, [2.5, 97.5]))
p_perm = (np.abs(perm) >= abs(b_obs)).mean()
print(f"cluster bootstrap ({len(boot)} converged): IRR 95% CI [{ci[0]:.3f}, {ci[1]:.3f}], "
      f"P(coef >= 0) = {(boot >= 0).mean():.3f}")
print(f"permutation test ({len(perm)} fits):       two-sided p = {p_perm:.4f}  "
      f"(asymptotic p = {m.pvalues['exits']:.1e})")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Cluster bootstrap (by fire): IRR per exit", "Permutation null vs observed coefficient"))
fig.add_trace(go.Histogram(x=np.exp(boot), nbinsx=60, marker_color="steelblue", showlegend=False), 1, 1)
fig.add_vline(x=1.0, line_color="gray", row=1, col=1)
fig.add_vline(x=float(np.exp(b_obs)), line_dash="dash", line_color="firebrick",
              annotation_text="observed", row=1, col=1)
fig.add_trace(go.Histogram(x=perm, nbinsx=60, marker_color="darkseagreen", showlegend=False), 1, 2)
fig.add_vline(x=float(b_obs), line_dash="dash", line_color="firebrick",
              annotation_text="observed", row=1, col=2)
fig.update_layout(height=420, title="Resampling-based inference")
fig.show()

/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fata

/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-f

/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/user/egress-fata

cluster bootstrap (965 converged): IRR 95% CI [0.785, 0.921], P(coef >= 0) = 0.001
permutation test (2000 fits):       two-sided p = 0.0350  (asymptotic p = 1.9e-06)


/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/home/user/egress-fatalities/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


## 6. The cumulative-sum artifact, demonstrated by simulation

§3 argued the paper's headline "threshold at ~6 exits" is an artifact of its y-axis:
`cum_per_fatal = sum(per_fatal) - cumsum(per_fatal)` — a **reverse cumulative sum** of the
per-capita rate, sorted by exits (their `sd03.r`). Here we prove it. We keep the real exit
distribution and the exact reverse-cumsum construction, but replace each community's fatality
rate with a **random uniform draw** — pure noise, no relationship to exits whatsoever — and
re-fit the same segmented breakpoint over 2,000 simulations.

If the threshold were real, random data should destroy it. It doesn't: the breakpoint lands in
the same place every time, because (a) a reverse cumulative sum can only descend, and (b) the
right-skewed exit distribution forces the bend wherever communities are densest — near the
median, regardless of the y-values.

In [12]:
from plotly.subplots import make_subplots
from scipy.stats import percentileofscore

rng = np.random.default_rng(7)
exits_sorted = np.sort(df.exits.values)        # real exit distribution, ascending
n = len(exits_sorted)

def segmented_breakpoint(x, y):
    """Best continuous two-segment (hinge) fit by SSR over candidate breaks — mimics R's `segmented`."""
    best = None
    for bp in np.linspace(x.min() + 1e-6, x.max() - 1e-6, 200):
        X = np.column_stack([np.ones_like(x), x, np.maximum(x - bp, 0)])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        ssr = ((y - X @ beta) ** 2).sum()
        if best is None or ssr < best[0]:
            best = (ssr, bp)
    return best[1]

N_SIM = 2000
curves, bps = [], []
for _ in range(N_SIM):
    u = rng.uniform(0, 1, n)                    # random "fatality rate", no trend
    cum = u.sum() - np.cumsum(u)               # authors' reverse cumulative sum
    curves.append(cum)
    bps.append(segmented_breakpoint(exits_sorted, cum))
curves = np.array(curves); bps = np.array(bps)

bp_med = np.median(bps)
bp_pct = np.median([percentileofscore(exits_sorted, b, kind="mean") for b in bps])
print(f"null breakpoint over {N_SIM} pure-noise runs: median {bp_med:.2f} exits "
      f"({bp_pct:.0f}th pct)   |   authors reported ~6.3 exits / 66.6th pct")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Reverse-cumulative curves from random data", "Where the breakpoint lands (2,000 runs)"))
for c in curves[rng.choice(N_SIM, 8, replace=False)]:
    fig.add_trace(go.Scatter(x=exits_sorted, y=c, mode="lines",
                             line=dict(color="lightgray", width=1), showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=exits_sorted, y=curves.mean(0), mode="lines", name="mean of runs",
                         line=dict(color="firebrick", width=3)), 1, 1)
fig.add_vline(x=bp_med, line_dash="dash", line_color="rebeccapurple", row=1, col=1)
fig.add_trace(go.Histogram(x=bps, nbinsx=40, marker_color="mediumpurple", showlegend=False), 1, 2)
fig.add_vline(x=6.3, line_dash="dash", line_color="firebrick",
              annotation_text="authors' 6.3", row=1, col=2)
fig.update_xaxes(title_text="egress exits (real distribution)", row=1, col=1)
fig.update_xaxes(title_text="fitted breakpoint (exits)", row=1, col=2)
fig.update_yaxes(title_text="reverse-cumulative (random) rate", row=1, col=1)
fig.update_layout(height=420, title="Pure-noise data reproduces the paper's threshold")
fig.show()

null breakpoint over 2000 pure-noise runs: median 5.72 exits (61th pct)   |   authors reported ~6.3 exits / 66.6th pct


## 7. Who dies — victim age distribution

The fatality workbook (`sd01.xlsx`) names individual victims with ages, in three free-text
formats in the `notes` column (`Name (74)`, `Name, 74`, `74-year-old Name`) plus a structured
`Age` column on the Tubbs sheet. `scripts/generate_histogram_data.py` extracts them all,
de-duplicates notes that are attached to two census places of the same fire, and writes
`data/fatality-ages.csv` — **293 victim ages**, ~93% of the 314 identified fatalities.

**Caveat:** ages come from journalistic obituaries cited in the `fatality information` column,
so completeness varies by fire — the large, well-reported fires (Lahaina, Camp, Eaton, Tubbs)
supply most of the data.

In [13]:
ages = pd.read_csv("data/fatality-ages.csv")
baseline = pd.read_csv("data/age-exposure-baseline.csv")
bands = baseline.decade_band.tolist()          # canonical decade order

counts = ages.decade_band.value_counts().reindex(bands, fill_value=0)
a = ages.age.to_numpy()
print(f"n = {len(a)} ages | median {np.median(a):.0f}, mean {a.mean():.1f}, "
      f"range {a.min()}–{a.max()} | 65+ {(a>=65).mean()*100:.0f}%  75+ {(a>=75).mean()*100:.0f}%  "
      f"<18 {(a<18).sum()}")

fig = go.Figure(go.Bar(x=bands, y=counts.values, marker_color="steelblue",
                       hovertemplate="%{x}: %{y} deaths<extra></extra>"))
fig.add_vline(x=np.median(a) / 10 - 0.5, line_dash="dash", line_color="firebrick",
              annotation_text=f"median {np.median(a):.0f}")
fig.update_layout(title="Wildfire fatalities by age band (n = 293)", height=420,
                  xaxis_title="age band", yaxis_title="number of fatalities",
                  annotations=[*fig.layout.annotations,
                      dict(x=0.02, y=0.95, xref="paper", yref="paper", showarrow=False,
                           text="70% of victims were 65 or older", font=dict(color="gray"))])
fig.show()

n = 293 ages | median 71, mean 67.7, range 4–101 | 65+ 70%  75+ 38%  <18 8


## 8. Exposure-adjusted: relative risk by age

Raw counts conflate "this age dies often" with "this age is common." To separate them we
compute an SMR-style **relative risk**: observed deaths in a band ÷ deaths *expected* if these
victims died at the national age structure's rate (`data/age-exposure-baseline.csv`). RR = 1
means a band dies exactly at its population share; above = over-represented. Error bars are
exact Poisson 95% CIs on the observed counts (wide where counts are tiny).

**Two caveats.** (1) The honest denominator is the age structure of *at-risk WUI communities*,
which skew older than the nation — so national exposure **overstates** the elderly excess
somewhat. (2) This is composition among the dead, conditional on a fatal fire — not a per-fire
hazard. Even so, the gradient is steep and monotonic: risk crosses 1 around age 60 and reaches
4–6× in the 70s–90s.

In [14]:
from scipy.stats import chi2

deaths = counts.values.astype(float)
expected = baseline.us_pop_share_pct.values / 100 * deaths.sum()
rr = deaths / expected

def poisson_ci(d):                              # exact Poisson 95% CI on a count
    lo = chi2.ppf(0.025, 2 * d) / 2 if d > 0 else 0.0
    hi = chi2.ppf(0.975, 2 * (d + 1)) / 2
    return lo, hi

ci = np.array([poisson_ci(d) for d in deaths])
lo, hi = ci[:, 0] / expected, ci[:, 1] / expected

summary = pd.DataFrame({"band": bands, "deaths": deaths.astype(int),
                        "expected": expected.round(1), "RR": rr.round(2),
                        "CI_low": lo.round(2), "CI_high": hi.round(2)}).set_index("band")
print(summary.to_string())

fig = go.Figure(go.Bar(
    x=bands, y=rr, marker_color=["firebrick" if v >= 1 else "lightslategray" for v in rr],
    error_y=dict(type="data", symmetric=False, array=hi - rr, arrayminus=rr - lo, color="dimgray"),
    hovertemplate="%{x}: RR %{y:.2f}<extra></extra>"))
fig.add_hline(y=1, line_dash="dash", line_color="rebeccapurple", annotation_text="expected (RR=1)")
fig.update_yaxes(type="log", title_text="relative risk (observed ÷ expected, log scale)")
fig.update_layout(title="Exposure-adjusted relative risk of wildfire death by age",
                  height=440, xaxis_title="age band")
fig.show()

       deaths  expected     RR  CI_low  CI_high
band                                           
0–9         4      34.3   0.12    0.03     0.30
10–19       4      36.9   0.11    0.03     0.28
20–29       5      39.3   0.13    0.04     0.30
30–39      11      39.6   0.28    0.14     0.50
40–49      12      35.7   0.34    0.17     0.59
50–59      30      37.8   0.79    0.54     1.13
60–69      68      34.0   2.00    1.55     2.54
70–79      94      22.3   4.22    3.41     5.17
80–89      48      10.5   4.55    3.36     6.03
90–99      15       2.5   6.02    3.37     9.93
100+        2       0.1  22.75    2.76    82.19


## 9. Exposure re-specification — structures destroyed and pre-fire age structure

Two further exposure variables per record (`scripts/fetch_exposure_data.py`): **structures
destroyed** in the fire (`data/fire-structures-destroyed.csv`, hand-curated and source-cited
per row — CAL FIRE incident pages / Top-20 list, FEMA/PDC, NPS, after-action reports) and the
**pre-fire population age structure** in 10-year bins from ACS 5-year table B01001, vintage
ending before the fire year (`data/place-age-distribution.csv`, written by the fetch script).

Four count-model specs. A is §1's corrected baseline. B adds log structures destroyed with the
population offset intact. C swaps the offset — deaths per *structure destroyed* instead of per
resident. D frees both elasticities and lets the data apportion exposure between population
and destruction.

Caveat: unlike population, structures destroyed is not pre-determined exposure — it is realized
in the same event as the deaths and shares their severity drivers (and is a fire-level total
repeated across census places of multi-place fires). B–D are severity-conditioning
specifications, not causal decompositions.

In [15]:
st = pd.read_csv("data/fire-structures-destroyed.csv")
dfx = df.merge(st[["Fire", "structures_destroyed"]], on="Fire", how="left")
dfx["lstruct"] = np.log(dfx.structures_destroyed)
dfx["lpop"] = np.log(dfx.population_corrected)


def nb_fit(formula, data, offset=None):
    """NB2 MLE; Nelder-Mead restart when BFGS stalls on the alpha ridge (spec C needs it)."""
    mod = smf.negativebinomial(formula, data, offset=offset)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # pre-restart Hessian warning
        res = mod.fit(disp=0)
    if not res.mle_retvals["converged"] or res.bse.isna().any():
        res = mod.fit(start_params=mod.fit(method="nm", maxiter=5000, disp=0).params, disp=0)
    return res


specs = {
    "A: exits | offset log(pop)": nb_fit("Identified_fatalities ~ exits", dfx, offset=dfx.lpop),
    "B: A + log(structures)": nb_fit("Identified_fatalities ~ exits + lstruct", dfx, offset=dfx.lpop),
    "C: exits | offset log(structures)": nb_fit("Identified_fatalities ~ exits", dfx, offset=dfx.lstruct),
    "D: exits + log(structures) + log(pop)": nb_fit("Identified_fatalities ~ exits + lstruct + lpop", dfx),
}

rows = []
for name, m9 in specs.items():
    for term in m9.params.index.drop(["Intercept", "alpha"]):
        b, p = m9.params[term], m9.pvalues[term]
        lo, hi = m9.conf_int().loc[term]
        rows.append({"spec": name, "term": term, "IRR": round(np.exp(b), 3),
                     "95% CI": f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]", "p": f"{p:.2g}",
                     "alpha": round(m9.params["alpha"], 2), "AIC": round(m9.aic, 1)})
pd.DataFrame(rows).set_index(["spec", "term"])

IRR          95% CI        p  \
spec                                  term                                      
A: exits | offset log(pop)            exits    0.837  [0.777, 0.900]  1.9e-06   
B: A + log(structures)                exits    0.831  [0.773, 0.894]  6.2e-07   
                                      lstruct  1.120  [0.912, 1.375]     0.28   
C: exits | offset log(structures)     exits    0.980  [0.892, 1.077]     0.68   
D: exits + log(structures) + log(pop) exits    0.918  [0.856, 0.983]    0.015   
                                      lstruct  1.346  [1.109, 1.634]   0.0026   
                                      lpop     1.541  [1.214, 1.956]  0.00038   

                                               alpha    AIC  
spec                                  term                   
A: exits | offset log(pop)            exits     1.05  228.4  
B: A + log(structures)                exits     1.00  229.3  
                                      lstruct   1.00  229.3  
C: exits | offset log(structures)     exits     1.35  241.9  
D: exits + log(structures) + log(pop) exits     0.70  213.4  
                                      lstruct   0.70  213.4  
                                      lpop      0.70  213.4

**Reading.** The per-capita exits effect (A) is unmoved by adding destruction as a covariate
(B: −17%/exit, p ≈ 6e-7; log structures n.s. with the population offset in place) — but the
*choice of exposure* matters enormously. Per structure destroyed (C), the exits effect vanishes
(IRR 0.98/exit, p = 0.68): low-exit communities do not lose more people per structure destroyed,
and exits are nearly orthogonal to destruction itself (r = 0.18 with log structures). With both
elasticities free (D; ΔAIC ≈ −15 vs A), fatalities scale with destruction (elasticity ≈ 0.30,
p = 0.003) and sub-linearly with population (≈ 0.43, p < 0.001), while the exits effect
attenuates by half, to −8%/exit (p = 0.015). Roughly half of the headline per-capita exits
effect is absorbed by how much of the community actually burned; what remains is an
independent, smaller egress signal.

### 9b. Pre-fire age structure (10-year bins)

`data/place-age-distribution.csv` comes from the Census API and must be generated where
outbound network is available: `python scripts/fetch_exposure_data.py` (see readme). The cell
below is self-gating — it reports what to do if the file is absent and runs the age models once
it exists. Spec E adds the 70+ population share (percentage points) to D; spec F adds the full
composition (8 bins, 30–39 as the omitted reference — shares are compositional, so one bin must
drop). F spends 12 parameters on 38 records: read it as descriptive, not confirmatory.

In [16]:
from pathlib import Path

BINS = ["0_9", "10_19", "20_29", "30_39", "40_49", "50_59", "60_69", "70_79", "80p"]
age_csv = Path("data/place-age-distribution.csv")
if not age_csv.exists():
    print("data/place-age-distribution.csv not found -> run `python scripts/fetch_exposure_data.py`\n"
          "(needs outbound access to api.census.gov), then re-run this cell.")
else:
    age_bins = pd.read_csv(age_csv)
    dfa = dfx.merge(age_bins[["Fire", "census_place"] + [f"share_{b}" for b in BINS]],
                    on=["Fire", "census_place"], how="left", validate="1:1")
    for b in BINS:
        dfa[f"pp_{b}"] = 100 * dfa[f"share_{b}"]  # percentage points of place population
    dfa["pp_70p"] = dfa.pp_70_79 + dfa.pp_80p

    E = nb_fit("Identified_fatalities ~ exits + lstruct + lpop + pp_70p", dfa)
    comp = " + ".join(f"pp_{b}" for b in BINS if b != "30_39")
    F = nb_fit(f"Identified_fatalities ~ exits + lstruct + lpop + {comp}", dfa)

    out = []
    for name, m9 in [("E: D + 70+ share", E), ("F: D + full composition (ref 30-39)", F)]:
        for term in m9.params.index.drop(["Intercept", "alpha"]):
            b, p = m9.params[term], m9.pvalues[term]
            lo, hi = m9.conf_int().loc[term]
            out.append({"spec": name, "term": term, "IRR": round(np.exp(b), 3),
                        "95% CI": f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]", "p": f"{p:.2g}",
                        "AIC": round(m9.aic, 1), "converged": m9.mle_retvals["converged"]})
    display(pd.DataFrame(out).set_index(["spec", "term"]))

IRR          95% CI        p  \
spec                                term                                       
E: D + 70+ share                    exits     0.914  [0.854, 0.979]    0.011   
                                    lstruct   1.346  [1.113, 1.628]   0.0022   
                                    lpop      1.558  [1.226, 1.980]  0.00029   
                                    pp_70p    0.967  [0.925, 1.011]     0.14   
F: D + full composition (ref 30-39) exits     0.919  [0.857, 0.985]    0.017   
                                    lstruct   1.488  [1.204, 1.839]  0.00023   
                                    lpop      1.242  [0.888, 1.738]     0.21   
                                    pp_0_9    0.846  [0.714, 1.003]    0.054   
                                    pp_10_19  0.878  [0.751, 1.027]      0.1   
                                    pp_20_29  0.903  [0.803, 1.015]    0.088   
                                    pp_40_49  0.910  [0.810, 1.022]     0.11   
                                    pp_50_59  0.861  [0.755, 0.981]    0.025   
                                    pp_60_69  0.875  [0.784, 0.976]    0.017   
                                    pp_70_79  0.884  [0.745, 1.048]     0.16   
                                    pp_80p    0.879  [0.749, 1.031]     0.11   

                                                AIC  converged  
spec                                term                        
E: D + 70+ share                    exits     213.3       True  
                                    lstruct   213.3       True  
                                    lpop      213.3       True  
                                    pp_70p    213.3       True  
F: D + full composition (ref 30-39) exits     219.4       True  
                                    lstruct   219.4       True  
                                    lpop      219.4       True  
                                    pp_0_9    219.4       True  
                                    pp_10_19  219.4       True  
                                    pp_20_29  219.4       True  
                                    pp_40_49  219.4       True  
                                    pp_50_59  219.4       True  
                                    pp_60_69  219.4       True  
                                    pp_70_79  219.4       True  
                                    pp_80p    219.4       True

**Reading (9b).** Neither age spec improves on D. The 70+ population share enters slightly
*negative* (IRR 0.967 per percentage point) and insignificant (p = 0.14, AIC unchanged at 213),
and the full composition is worse by AIC (219 vs 213) with every bin sitting a similar ~9–15%
below the 30–39 reference — a pattern that re-parametrizes level differences across places, not
an age gradient. The individual-level finding (§7–8: fatality risk concentrates overwhelmingly
in 70+) does not surface as a place-level compositional effect across n = 38 communities: *which*
places suffer deadly fires is governed by destruction, size and egress; *who* dies within them is
governed by age. The exits effect is stable at ≈ −8 to −9% per exit (p ≈ 0.01–0.02) in every
exposure-conditioned spec.

## Conclusions

- **The effect direction is real and robust:** ~16% lower fatality rate per additional egress road
  (IRR 0.837, 95% CI [0.777, 0.900], p ≈ 2×10⁻⁶) on corrected denominators; ~19% on the original ones.
- **No threshold:** the decline is smooth; the paper's "~6 exits" breakpoint was an artifact of
  regressing a reverse cumulative sum on its own sort key.
- **The paper's per-capita OLS does not survive the denominator corrections** (p 0.036 → 0.060).
- **No single community drives the result** — the jackknife coefficient stays in [−0.196, −0.168],
  and the most influential point (Eaton/Altadena) *strengthens* the effect when dropped.

**Robustness (§5):**

- **The effect direction survives every check** — alternative likelihoods (including zero-truncated NB:
  −14.8%/exit, p = 0.027), block deletions (dropping Lahaina+Paradise, the corrected rows, or tiny
  CDPs barely moves it), a cluster bootstrap by fire (IRR 95% CI [0.785, 0.921]), and a free
  population elasticity (attenuates to −9.4%/exit, p = 0.011, still negative).
- **But the headline p-value should be read as ~0.04, not 10⁻⁶.** The permutation test — the honest
  small-sample null — gives p ≈ 0.04. Significant, not overwhelming.
- **Still no threshold:** a hinge at 6 exits is an AIC tie with linear and its term is not
  significant (p = 0.155).
- Still observational, ~35 communities, selection on fatal events, exits confounded with community
  type — **no causal or policy reading is warranted.**

**Structural artifact & demographics (§6–8):**

- **The "~6 exit threshold" is reproduced by pure noise (§6).** Feeding random uniform rates
  through the authors' reverse-cumulative construction lands the segmented breakpoint at a median
  of ~5.7 exits across 2,000 runs — statistically indistinguishable from their reported 6.3. The
  threshold measures the shape of the exit distribution, not fatalities.
- **Deaths are overwhelmingly elderly (§7–8).** Median victim age 71; 70% were 65+. Exposure-
  adjusted against the national age structure, relative risk crosses 1 near age 60 and rises to
  4–6× in the 70s–90s — the mechanism (mobility-limited evacuation) behind the egress signal,
  though the WUI denominator caveat means the elderly excess is somewhat overstated here.